In [8]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser, CommaSeparatedListOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableParallel

from config import OPEN_AI_KEY as API_KEY


chat = ChatOpenAI(model="gpt-5.6-luna", seed=365, temperature=0, api_key=API_KEY)
str_parser = StrOutputParser()
list_parser = CommaSeparatedListOutputParser()
pass_through = RunnablePassthrough()

In [3]:
msg_tool = "What are the five most important tool a {job} needs? Answer only by listing the tools."
msg_strategy = "Considering the tools provided, develop a strategy (not more than 300 words) for effectively learning and mastering them: {tools}"
chat_template_tools = ChatPromptTemplate.from_template(msg_tool)
chat_template_strategy = ChatPromptTemplate.from_template(msg_strategy)
chain_long = (chat_template_tools | chat | str_parser | {"tools": pass_through} | chat_template_strategy | chat | str_parser)

In [6]:
chain_long.get_graph().print_ascii()

     +-------------+       
     | PromptInput |       
     +-------------+       
            *              
            *              
            *              
  +--------------------+   
  | ChatPromptTemplate |   
  +--------------------+   
            *              
            *              
            *              
      +------------+       
      | ChatOpenAI |       
      +------------+       
            *              
            *              
            *              
   +-----------------+     
   | StrOutputParser |     
   +-----------------+     
            *              
            *              
            *              
+-----------------------+  
| StrOutputParserOutput |  
+-----------------------+  
            *              
            *              
            *              
     +-------------+       
     | Passthrough |       
     +-------------+       
            *              
            *              
            *       

In [44]:
list_instruction = list_parser.get_format_instructions()
msg_books = "Suggest three of the best intermediate-level {programming language} books. " + list_instruction
msg_projects = "Suggest three interesting {programming language} projects suitable for intermediate-level programmers. " + list_instruction
msg_time = "I'm an intermediate level programmer. Consider the following literature: {books}. Also consider the following projects: {projects}. Roughly, how much time would it take me to complete the literature and the projects?"
chat_template_books = ChatPromptTemplate.from_template(msg_books)
chat_template_projects = ChatPromptTemplate.from_template(msg_projects)
chat_template_time = ChatPromptTemplate.from_template(msg_time)

In [13]:
chain_books = chat_template_books | chat | str_parser
chain_projects = chat_template_projects | chat | str_parser
chain_parallel = RunnableParallel({"books": chain_books, "projects": chain_projects})

In [16]:
result = chain_parallel.invoke({"programming language": "Python"})

In [17]:
result

{'books': 'Fluent Python by Luciano Ramalho, Effective Python by Brett Slatkin, Python Cookbook by David Beazley and Brian K. Jones',
 'projects': 'Personal finance tracker, Real-time collaborative whiteboard, Python package dependency visualizer'}

In [18]:
%%time
result_1 = chain_parallel.invoke({"programming language": "Rust"})
result_2 = chain_parallel.invoke({"programming language": "C++"})
result_3 = chain_parallel.invoke({"programming language": "C#"})
result_4 = chain_parallel.invoke({"programming language": "Javascript"})
print(result_1, result_2, result_3, result_4)

{'books': 'Programming Rust: Fast, Safe Systems Development, Rust for Rustaceans: An Intermediate Guide to the Rust Programming Language, Effective Rust: 35 Specific Ways to Improve Your Rust Code', 'projects': 'A command-line task manager with file persistence, A multithreaded web crawler with rate limiting, A simple 2D game engine using an ECS architecture'} {'books': 'Effective Modern C++ by Scott Meyers, C++ Primer by Stanley B. Lippman et al., A Tour of C++ by Bjarne Stroustrup', 'projects': 'Personal finance tracker, Multiplayer chat application, 2D game engine'} {'books': 'Effective C# by Bill Wagner, C# 12 in a Nutshell by Joseph Albahari and Ben Albahari, Pro C# 12 and .NET 8 by Andrew Troelsen and Philip Japikse', 'projects': 'Personal finance tracker, Real-time multiplayer chat application, 2D game with a level editor'} {'books': 'You Don’t Know JS Yet by Kyle Simpson, Effective JavaScript by David Herman, JavaScript: The Definitive Guide by David Flanagan', 'projects': 'Rea

In [41]:
%%time
result_5 = chain_parallel.invoke({"programming language": "Django"})
result_6 = chain_parallel.invoke({"programming language": "AI Engineering"})

CPU times: user 53.2 ms, sys: 0 ns, total: 53.2 ms
Wall time: 6.65 s


In [42]:
result_6

{'books': 'AI Engineering – Chip Huyen, Designing Machine Learning Systems – Chip Huyen, Machine Learning Engineering – Andriy Burkov',
 'projects': 'RAG-powered personal knowledge assistant, Real-time computer vision quality inspector, LLM-based code review and test generation tool'}

In [43]:
chain_parallel.get_graph().print_ascii()

            +-------------------------------+              
            | Parallel<books,projects>Input |              
            +-------------------------------+              
                   ***               ***                   
                ***                     ***                
              **                           **              
+--------------------+              +--------------------+ 
| ChatPromptTemplate |              | ChatPromptTemplate | 
+--------------------+              +--------------------+ 
           *                                   *           
           *                                   *           
           *                                   *           
    +------------+                      +------------+     
    | ChatOpenAI |                      | ChatOpenAI |     
    +------------+                      +------------+     
           *                                   *           
           *                            

In [45]:
# chain_time = {"books": chain_books, "projects": chain_projects} | chat_template_time | chat | str_parser  # Would also work
chain_time = chain_parallel | chat_template_time | chat | str_parser

In [46]:
chain_time.get_graph().print_ascii()

            +-------------------------------+              
            | Parallel<books,projects>Input |              
            +-------------------------------+              
                   ***               ***                   
                ***                     ***                
              **                           **              
+--------------------+              +--------------------+ 
| ChatPromptTemplate |              | ChatPromptTemplate | 
+--------------------+              +--------------------+ 
           *                                   *           
           *                                   *           
           *                                   *           
    +------------+                      +------------+     
    | ChatOpenAI |                      | ChatOpenAI |     
    +------------+                      +------------+     
           *                                   *           
           *                            

In [49]:
result_7 = chain_time.invoke({"programming language": "AI Engineering"})

In [51]:
print(str(result_7))

Assuming you are comfortable with Python, APIs, basic software engineering, and introductory machine learning—but are not already experienced in production ML—this is roughly a **6–12 month part-time effort**.

## Estimated time

| Component | Basic completion | Solid portfolio-quality completion |
|---|---:|---:|
| *AI Engineering* — Chip Huyen | 20–35 hours | 35–50 hours |
| *Designing Machine Learning Systems* — Chip Huyen | 20–35 hours | 35–50 hours |
| *Machine Learning Engineering* — Andriy Burkov | 25–45 hours | 40–60 hours |
| Document question-answering system | 40–70 hours | 80–140 hours |
| Real-time image anomaly detection | 50–90 hours | 100–180 hours |
| Personalized recommendation engine | 60–120 hours | 120–220 hours |
| Documentation, testing, deployment, debugging | 30–60 hours | 60–120 hours |
| **Total** | **245–455 hours** | **470–820 hours** |

The lower end assumes you use existing models and managed services. The higher end assumes you build meaningful evaluatio

### Runnable Lambda

In [57]:
from langchain_core.runnables import RunnableLambda, chain

find_sum = lambda x: sum(x)
find_square = lambda x: x*x
print(find_sum([2,3,4]), find_square(8))

9 64


In [58]:
runnable_sum = RunnableLambda(lambda x: sum(x))
runnable_square = RunnableLambda(lambda x: x ** 2)
runnable_square.invoke(8)

64

In [59]:
chain_1 = runnable_sum | runnable_square
chain_1.invoke([1, 2, 5])

64

In [60]:
chain_1.get_graph().print_ascii()

+-------------+  
| LambdaInput |  
+-------------+  
        *        
        *        
        *        
   +--------+    
   | Lambda |    
   +--------+    
        *        
        *        
        *        
   +--------+    
   | Lambda |    
   +--------+    
        *        
        *        
        *        
+--------------+ 
| LambdaOutput | 
+--------------+ 


### @chain Decorator

In [62]:
def find_sum(x):
    return sum(x)

def find_square(x):
    return x ** 2

chain_2 = RunnableLambda(find_sum) | RunnableLambda(find_square)
chain_2.invoke([2, 3])

25

In [64]:
@chain
def runnable_sum(x):
    return sum(x)

@chain
def runnable_square(x):
    return x ** 2

In [66]:
print(type(find_square))
print(type(runnable_square))

<class 'function'>
<class 'langchain_core.runnables.base.RunnableLambda'>
